In [1]:
import pandas as pd
import numpy as np
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
export_dir = os.getcwd()
from pathlib import Path
import pickle

from collections import defaultdict
import time
import torch
import torch.nn as nn
import copy
import torch.nn.functional as F
import optuna
import logging
import matplotlib.pyplot as plt
import ipynb
import importlib
import sys
from pathlib import Path
from collections import Counter

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split 
## Get project root (one level up from notebooks/)
#project_root = Path.cwd().parent
#sys.path.append(str(project_root))


In [31]:
data_name = "LASTFM" ### Can be ML1M, Yahoo, Pinterest
recommender_name = "VAE" ## Can be MLP, C, MLP_model, GMF_model, NCF
DP_DIR = Path("data/", data_name) 
export_dir = Path(os.getcwd())
files_path = Path(export_dir, DP_DIR)
checkpoints_path = Path(export_dir, "checkpoints")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#device='cpu'

In [3]:
## Load the data
train_data = pd.read_csv(Path(files_path,f'Train_Data_{data_name}_New.csv'), index_col=0)

test_data = pd.read_csv(Path(files_path,f'Test_Data_{data_name}_New.csv'), index_col=0)

static_test_data = pd.read_csv(Path(files_path,f'Static_Test_Data_{data_name}_New.csv'), index_col=0)

## Load the items data and the popularity dictionary 
# items_data: meta features like genre

items_data = pd.read_csv(Path(files_path,f'Items_Data_{data_name}_New.csv'), index_col=0)
with open(Path(files_path,f'pop_dict_{data_name}.pkl'), 'rb') as f:
    pop_dict = pickle.load(f)

    
train_array = train_data.to_numpy()
test_array = test_data.to_numpy()
num_items=len(train_data.iloc[:,:-1].columns)
num_users=np.concatenate([train_array,test_array],axis=0).shape[0]

items_array = np.eye(num_items)
all_items_tensor = torch.Tensor(items_array).to(device)
all_array=np.concatenate((train_array, test_array))

## df for training and tes datasets without the users index
all_df_data=pd.concat([train_data.iloc[:,:-1],test_data.iloc[:,:-1]], axis=0 )
all_df_data.columns=all_df_data.columns.astype(int)

In [6]:
output_type_dict = {
    "VAE":"multiple",
    "MLP":"single"
}

output_type = output_type_dict[recommender_name] ### Can be single, multiple

In [7]:
pop_array = np.zeros(len(pop_dict))
for key, value in pop_dict.items():
    pop_array[key] = value

## Recommendations import

In [8]:
from ipynb.fs.defs.recommenders_architecture import *
importlib.reload(ipynb.fs.defs.recommenders_architecture)
from ipynb.fs.defs.recommenders_architecture import *

## Help functions import

In [9]:
from ipynb.fs.defs.help_functions import *
importlib.reload(ipynb.fs.defs.help_functions)
from ipynb.fs.defs.help_functions import *

In [10]:
output_type_dict = {
    "VAE":"multiple",
    "MLP":"single"
}


In [11]:
kw_dict = {'device':device,
          'num_items': num_items,
          'pop_array':pop_array,
          'all_items_tensor':all_items_tensor,
          'static_test_data':static_test_data,
          'items_array':items_array,
          'output_type':output_type,
          'recommender_name':recommender_name}

## Training the MLP recommender

In [12]:
train_losses_dict = {}
test_losses_dict = {}
HR10_dict = {}

def MLP_objective(trial):
    
    #lr = trial.suggest_float('learning_rate', 0.001, 0.01)
    lr=0.001
    #batch_size = trial.suggest_categorical('batch_size', [256, 512, 1024])
    batch_size=512
    #hidden_dim = trial.suggest_categorical('hidden_dim', [64, 128, 256, 512])
    hidden_dim=128
    beta=0.8670671663707932
    #beta = trial.suggest_float('beta', 0, 4) # hyperparameter that weights the different loss terms
    epochs = 50
    model = MLP(hidden_dim, **kw_dict)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses = []
    test_losses = []
    hr10 = []
    
    print(f'======================== new run - {recommender_name} ========================')
    logger.info(f'======================== new run - {recommender_name} ========================')
    
    num_training = train_data.shape[0]
    num_batches = int(np.ceil(num_training / batch_size))

    
    for epoch in range(epochs):
        train_matrix = sample_indices(train_data.copy(), **kw_dict)
        #print(f"train_matrix shape{train_matrix.shape}")
        perm = np.random.permutation(num_training)
        loss = []
        train_pos_loss=[]
        train_neg_loss=[]
        if epoch!=0 and epoch%10 == 0: # decrease the learning rate every 10 epochs
            lr = 0.1*lr
            optimizer.lr = lr
        
        for b in range(num_batches):
            optimizer.zero_grad()
            if (b + 1) * batch_size >= num_training:
                batch_idx = perm[b * batch_size:]
            else:
                batch_idx = perm[b * batch_size: (b + 1) * batch_size]    
            batch_matrix = torch.FloatTensor(train_matrix[batch_idx,:-2]).to(device)

            batch_pos_idx = train_matrix[batch_idx,-2]
            batch_neg_idx = train_matrix[batch_idx,-1]
            #print(f"items_array{items_array.shape}")
            batch_pos_items = torch.Tensor(items_array[batch_pos_idx]).to(device)
            batch_neg_items = torch.Tensor(items_array[batch_neg_idx]).to(device)

            #print(f"batch_matrix {batch_matrix.shape} and batch_pos_items {batch_pos_items.shape} ")

            pos_output = torch.diagonal(model(batch_matrix, batch_pos_items))
            #print('done')
            neg_output = torch.diagonal(model(batch_matrix, batch_neg_items))
            
            # MSE loss
            pos_loss = torch.mean((torch.ones_like(pos_output)-pos_output)**2)
            neg_loss = torch.mean((neg_output)**2)
            
            batch_loss = pos_loss + beta*neg_loss
            batch_loss.backward()
            optimizer.step()
            
            loss.append(batch_loss.item())
            train_pos_loss.append(pos_loss.item())
            train_neg_loss.append(neg_loss.item())
            
        print(f'train pos_loss = {np.mean(train_pos_loss)}, neg_loss = {np.mean(train_neg_loss)}')    
        train_losses.append(np.mean(loss))
        torch.save(model.state_dict(), Path(checkpoints_path, f'MLP_{data_name}_{round(lr,4)}_{batch_size}_{trial.number}_{epoch}.pt'))


        model.eval()
        test_matrix = np.array(static_test_data)
        test_tensor = torch.Tensor(test_matrix[:,:-2]).to(device)
        
        test_pos = test_matrix[:,-2]
        test_neg = test_matrix[:,-1]
        
        row_indices = np.arange(test_matrix.shape[0])
        test_tensor[row_indices,test_pos] = 0
        
        pos_items = torch.Tensor(items_array[test_pos]).to(device)
        neg_items = torch.Tensor(items_array[test_neg]).to(device)
        
        pos_output = torch.diagonal(model(test_tensor, pos_items).to(device))
        neg_output = torch.diagonal(model(test_tensor, neg_items).to(device))
        
        pos_loss = torch.mean((torch.ones_like(pos_output)-pos_output)**2)
        neg_loss = torch.mean((neg_output)**2)
        print(f'test pos_loss = {pos_loss}, neg_loss = {neg_loss}')
        
        hit_rate_at_10, hit_rate_at_50, hit_rate_at_100, MRR, MPR = recommender_evaluations(model, **kw_dict)
        hr10.append(hit_rate_at_10) # metric for monitoring
        print(f" epoch {epoch} hit_rate_at_10 {hit_rate_at_10} hit_rate_at_50 {hit_rate_at_50} hit_rate_at_100 {hit_rate_at_100} MRR {MRR} MPR {MPR}")
        
        test_losses.append(-hit_rate_at_10)
        if epoch>5: # early stop if the HR@10 decreases for 4 epochs in a row
            if test_losses[-2]<=test_losses[-1] and test_losses[-3]<=test_losses[-2] and test_losses[-4]<=test_losses[-3]:
                logger.info(f'Early stop at trial with batch size = {batch_size} and lr = {lr}. Best results at epoch {np.argmin(test_losses)} with value {np.min(test_losses)}')
                train_losses_dict[trial.number] = train_losses
                test_losses_dict[trial.number] = test_losses
                HR10_dict[trial.number] = hr10
                return max(hr10)
            
    logger.info(f'Stop at trial with batch size = {batch_size} and lr = {lr}. Best results at epoch {np.argmin(test_losses)} with value {np.min(test_losses)}')
    train_losses_dict[trial.number] = train_losses
    test_losses_dict[trial.number] = test_losses
    HR10_dict[trial.number] = hr10
    return max(hr10)

In [13]:
train_array=train_array[:,:-1]

In [14]:
test_array

array([[    0,     0,     0, ...,     0,     0, 19704],
       [    0,     0,     0, ...,     0,     0, 19705],
       [    0,     0,     0, ...,     0,     0, 19706],
       ...,
       [    0,     0,     0, ...,     0,     0, 24628],
       [    0,     0,     0, ...,     0,     0, 24629],
       [    0,     0,     0, ...,     0,     0, 24630]])

## Training VAE 

In [29]:
train_losses_dict = {}
test_losses_dict = {}
HR10_dict = {}


VAE_config= {
"enc_dims": [256,64],
"dropout": 0.5,
"anneal_cap": 0.2,
"total_anneal_steps": 200000
}

def VAE_objective(trial):
    
    lr = 0.01
    
    batch_size = 512
    epochs = 20
    model = VAE(VAE_config ,**kw_dict)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses = []
    test_losses = []
    hr10 = []
    print('======================== new run ========================')
    logger.info('======================== new run ========================')
    
    for epoch in range(epochs):
        if epoch!=0 and epoch%10 == 0:
            lr = 0.1*lr
            optimizer.lr = lr
        loss = model.train_one_epoch(train_array, optimizer, batch_size)
        train_losses.append(loss)
        torch.save(model.state_dict(), Path(checkpoints_path, f'VAE_{data_name}_{trial.number}_{epoch}_{round(lr,4)}_{batch_size}.pt'))


        model.eval()
        test_matrix = static_test_data.to_numpy()
        test_tensor = torch.Tensor(test_matrix[:,:-2]).to(device)
        test_pos = test_matrix[:,-2]
        test_neg = test_matrix[:,-1]
        row_indices = np.arange(test_matrix.shape[0])
        test_tensor[row_indices,test_pos] = 0
        output = model(test_tensor).to(device)
        pos_loss = -output[row_indices,test_pos].mean()
        
        neg_loss = output[row_indices, test_neg].mean()

        
        hit_rate_at_10, hit_rate_at_50, hit_rate_at_100, MRR, MPR = recommender_evaluations(model, **kw_dict)
        hr10.append(hit_rate_at_10)
        print(f" epoch {epoch} hit_rate_at_10 {hit_rate_at_10} hit_rate_at_50 {hit_rate_at_50} hit_rate_at_100 {hit_rate_at_100} MRR {MRR} MPR {MPR}")
        
        test_losses.append(pos_loss.item())
        if epoch>5:
            if test_losses[-2]<test_losses[-1] and test_losses[-3]<test_losses[-2] and test_losses[-4]<test_losses[-3]:
                logger.info(f'Early stop at trial with batch size = {batch_size} and lr = {lr}. Best results at epoch {np.argmin(test_losses)} with value {np.min(test_losses)}')
                train_losses_dict[trial.number] = train_losses
                test_losses_dict[trial.number] = test_losses
                HR10_dict[trial.number] = hr10
                return max(hr10)
    
    logger.info(f'Stop at trial with batch size = {batch_size} and lr = {lr}. Best results at epoch {np.argmin(test_losses)} with value {np.min(test_losses)}')
    train_losses_dict[trial.number] = train_losses
    test_losses_dict[trial.number] = test_losses
    HR10_dict[trial.number] = hr10
    return max(hr10)

## Save logs and using optuna

In [32]:
logger = logging.getLogger()

logger.setLevel(logging.INFO)  # Setup the root logger.
logger.addHandler(logging.FileHandler(f"{recommender_name}_{data_name}_Optuna.log", mode="w"))


optuna.logging.enable_propagation()  # Propagate logs to the root logger.
optuna.logging.disable_default_handler()  # Stop showing logs in sys.stderr.

study = optuna.create_study(direction='maximize')

logger.info("Start optimization.")

if recommender_name == 'MLP':
    study.optimize(MLP_objective, n_trials=1) 
elif recommender_name == 'VAE':
    study.optimize(VAE_objective, n_trials=1) 

with open(f"{recommender_name}_{data_name}_Optuna.log") as f:
    assert f.readline().startswith("A new study created")
    assert f.readline() == "Start optimization.\n"
    
    
# Print best hyperparameters and corresponding metric value
print("Best hyperparameters: {}".format(study.best_params))
print("Best metric value: {}".format(study.best_value))

======================== new run ========================
 epoch 0 hit_rate_at_10 0.0014207428455449563 hit_rate_at_50 0.006697787700426223 hit_rate_at_100 0.01603409782829308 MRR 0.0 MPR 49.7552572540871
 epoch 1 hit_rate_at_10 0.00040592652729855896 hit_rate_at_50 0.005682971382179825 hit_rate_at_100 0.013598538664501725 MRR 0.0 MPR 49.82892083470907
 epoch 2 hit_rate_at_10 0.00040592652729855896 hit_rate_at_50 0.006900750964075502 hit_rate_at_100 0.0158311345646438 MRR 0.0 MPR 49.75933254926529
 epoch 3 hit_rate_at_10 0.0006088897909478384 hit_rate_at_50 0.006088897909478384 hit_rate_at_100 0.01522224477369596 MRR 0.0 MPR 48.323352045576215
 epoch 4 hit_rate_at_10 0.02009336310127867 hit_rate_at_50 0.05378526486705906 hit_rate_at_100 0.07083417901359854 MRR 0.004262228536634869 MPR 37.15203699406214
 epoch 5 hit_rate_at_10 0.02313781205601786 hit_rate_at_50 0.08077937893241323 hit_rate_at_100 0.13578242338136798 MRR 0.004262228536634869 MPR 29.192094617872865
 epoch 6 hit_rate_at_10

In [19]:
static_test_data

,0,1,2,3,4,5,6,7,8,9,...,6576,6577,6578,6579,6580,6581,6582,6583,pos,neg
19704,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,224,4768
19705,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,4403,2093
19706,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2477,417
19707,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,6192,4640
19708,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,230,2751
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24626,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,812,4604
24627,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,5708,6055
24628,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,6065,3743
24629,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1304,1560


## Loading recommender

In [ ]:
output_type_dict = {
    "VAE":"multiple",
    "MLP":"single"
}


recommender_path_dict = {
    ("ML1M","VAE"): Path(checkpoints_path, "VAE_ML1M_17.pt"),
    ("ML1M","MLP"):Path(checkpoints_path, "MLP_ML1M_0.0_512_0_42.pt"),
    
    ("Yahoo","VAE"): Path(checkpoints_path, "VAE_Yahoo_0.0001_128_13.pt"),
    ("Yahoo","MLP"):Path(checkpoints_path, "MLP2_Yahoo_0.0083_128_1.pt"),

    ("Pinterest","VAE"): Path(checkpoints_path, "VAE_Pinterest_0.0002_32_12.pt"),
    ("Pinterest","MLP"):Path(checkpoints_path, "MLP_Pinterest_0.0062_512_21_0.pt")
    
}

hidden_dim_dict = {
    ("ML1M","VAE"): None,
    ("ML1M","MLP"): 128,

    ("Yahoo","VAE"): None,
    ("Yahoo","MLP"):32,
    
    ("Pinterest","VAE"): None,
    ("Pinterest","MLP"):512

}

output_type = output_type_dict[recommender_name] ### Can be single, multiple
hidden_dim = hidden_dim_dict[(data_name,recommender_name)]
recommender_path = recommender_path_dict[(data_name,recommender_name)]

In [ ]:
def load_recommender():
    if recommender_name=='MLP':
        recommender = MLP(hidden_dim, **kw_dict)
    elif recommender_name=='VAE':
        recommender = VAE(VAE_config, **kw_dict)

    recommender_checkpoint = torch.load(Path(checkpoints_path, recommender_path))
    recommender.load_state_dict(recommender_checkpoint)
    recommender.eval()
    for param in recommender.parameters():
        param.requires_grad= False
    return recommender
    
recommender = load_recommender()

## Recommenddations generation

In [ ]:
train_data['user_id']=train_data.index
test_data['user_id']=test_data.index
train_array=train_data.to_numpy()
test_array=test_data.to_numpy()
all_array=np.concatenate([train_array,test_array])

In [ ]:
def recommendations_generation(input_array):
    topk_itms = {}

    for i in range(input_array.shape[0]):
        user_index = input_array[i][-1]
        user_tensor = torch.Tensor(input_array[i][:-1]).to(device)
        
        topk_itms[user_index] = get_user_recommended_item(user_tensor, recommender, **kw_dict).cpu().numpy()
    return topk_itms

topk_itms=recommendations_generation(all_array)

In [ ]:
topk_itms

In [ ]:
orig_model_recommends


In [ ]:
orig_model_recommends=copy.deepcopy(topk_itms)

## Bias calculation

In [ ]:
Topk_itms=sorted(pop_dict.items(), key=lambda x:x[1], reverse=True)
#Topk_itms=dict(Topk_itms[0:100])
indx, values=zip(*Topk_itms[0:100])

In [ ]:
from itertools import chain
all_recommendations=list(chain.from_iterable(list(topk_itms.values())))
multual_itms=set(indx).intersection(all_recommendations)
count = sum(item in multual_itms for item in all_recommendations)
Total_exposure=count/len(all_recommendations)
print(f"Total_exposure is {Total_exposure}")

In [ ]:
from collections import Counter
count_itms=Counter(all_recommendations)